# Step 2 — Week 1: LLM Fundamentals

## Goals
- Understand tokenization and context window limits.
- Experiment with prompt formats: role prompting, few-shot examples, constraints.
- Track how `temperature` and `top_p` affect output consistency.

## Prerequisites
- Ollama installed and running: `ollama serve`
- Model pulled: `ollama pull llama3.2`
- Run the Setup cell below to install dependencies.

## Setup

In [1]:
%pip install requests tiktoken -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import json
import time

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"

def ollama_generate(prompt: str, system: str = "You respond like a funny black comedy style actor", temperature: float = 0.7,
                    top_p: float = 0.9, model: str = MODEL) -> str:
    body = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": top_p,
        },
    }
    if system:
        body["system"] = system

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=body,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["response"].strip()

# Smoke test
print(ollama_generate("Say: ready", temperature=1.0))

(sarcastically) Oh, joy. I was just sitting around twiddling my thumbs waiting for someone to say "ready" so we can embark on a thrilling adventure of... (pauses for comedic effect)... doing absolutely nothing. Bring it on! *cracks knuckles*


---
## Part 1 — Tokenization & Context Limits

### What is a token?
LLMs don't read text character-by-character — they operate on **tokens**.
A token is roughly 3–4 characters or 0.75 words on average in English.

| Text | Approx. tokens |
|---|---|
| `"Hello world"` | 2 |
| `"Retrieval-Augmented Generation"` | 5–6 |
| 1 page of English text (~500 words) | ~650 tokens |
| `llama3.2` context window | **128 k tokens** |

### Why does it matter?
- Every model has a **context window** — a hard cap on input + output tokens combined.
- Exceeding it silently truncates input or raises an error.
- Longer prompts = slower inference + higher cost (for paid APIs).

In [ ]:
import tiktoken

# tiktoken is OpenAI's tokenizer; useful as a close approximation for most LLMs
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / llama-style tokenizer

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def show_tokens(text: str) -> None:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"Text    : {repr(text)}")
    print(f"Tokens  : {tokens}")
    print(f"Decoded : {decoded}")
    print(f"Count   : {len(tokens)}")

show_tokens("Hello world")
print()
show_tokens("Retrieval-Augmented Generation improves LLM accuracy.")
print()
show_tokens("🤖 AI is transforming the world.")

### 1b) Tokenization edge cases
Understanding how tokenization splits words is critical for prompt design.

In [ ]:
edge_cases = [
    "tokenization",
    "un-tokenization",
    "TOKENIZATION",
    "token\nization",
    "12345678",
    "$100,000.00",
    "     leading spaces",
    "<|endoftext|>",
]

print(f"{'Text':<30} {'Tokens':>8} {'Splits'}")
print("-" * 70)
for text in edge_cases:
    tokens = enc.encode(text)
    splits = [enc.decode([t]) for t in tokens]
    print(f"{repr(text):<30} {len(tokens):>8}  {splits}")

### 1c) Context window budget calculator
Know exactly how much room you have before even sending the prompt.

In [ ]:
MODEL_CONTEXT_WINDOWS = {
    "llama3.2":        128_000,
    "llama3.1":        128_000,
    "mistral":          32_000,
    "gemma2":            8_192,
    "phi3":            128_000,
}

def context_budget(system: str, prompt: str, model: str = MODEL,
                   reserved_for_output: int = 512) -> None:
    ctx_limit = MODEL_CONTEXT_WINDOWS.get(model, 4_096)
    sys_tokens   = count_tokens(system)
    prompt_tokens = count_tokens(prompt)
    used  = sys_tokens + prompt_tokens + reserved_for_output
    remaining = ctx_limit - used

    print(f"Model context window : {ctx_limit:>10,} tokens")
    print(f"System message       : {sys_tokens:>10,} tokens")
    print(f"User prompt          : {prompt_tokens:>10,} tokens")
    print(f"Reserved for output  : {reserved_for_output:>10,} tokens")
    print(f"Total used           : {used:>10,} tokens")
    print(f"Remaining headroom   : {remaining:>10,} tokens")
    if remaining < 0:
        print("⚠  WARNING: prompt exceeds context window!")
    else:
        pct = 100 * used / ctx_limit
        print(f"Context utilisation  : {pct:>9.1f}%")

system_msg = "You are a concise Python tutor."
user_msg   = "Explain list comprehensions in Python with three examples."

context_budget(system_msg, user_msg, model="llama3.2")

---
## Part 2 — Prompt Formats

How you structure a prompt dramatically changes output quality.
We compare four patterns:

| Pattern | Description |
|---|---|
| **Zero-shot** | Task only, no examples or persona |
| **Role prompting** | Assign a persona via system message |
| **Few-shot** | Provide 2–3 input/output examples before the question |
| **Constrained** | Explicit output format rules embedded in the prompt |

### 2a) Zero-shot — baseline

In [ ]:
TASK = "Classify the sentiment of this review: 'The product broke after two days. Terrible quality.'"

zero_shot = ollama_generate(TASK, temperature=0.0)
print("=== Zero-shot ===")
print(zero_shot)

### 2b) Role prompting — assign a persona via system message

In [ ]:
role_system = (
    "You are a sentiment analysis engine. "
    "You only output a single word: POSITIVE, NEGATIVE, or NEUTRAL. "
    "Nothing else."
)

role_result = ollama_generate(TASK, system=role_system, temperature=0.0)
print("=== Role prompting ===")
print(role_result)

### 2c) Few-shot — provide examples before the real query

In [ ]:
few_shot_prompt = """Classify sentiment. Reply with only one word: POSITIVE, NEGATIVE, or NEUTRAL.

Review: "Absolutely love this! Best purchase this year."
Sentiment: POSITIVE

Review: "It works, does what it says. Nothing special."
Sentiment: NEUTRAL

Review: "Complete waste of money. Stopped working in a week."
Sentiment: NEGATIVE

Review: "The product broke after two days. Terrible quality."
Sentiment:"""

few_shot_result = ollama_generate(few_shot_prompt, temperature=0.0)
print("=== Few-shot ===")
print(few_shot_result)

### 2d) Constrained output — strict format rules in the prompt

In [ ]:
constrained_prompt = """Analyse the review below. Respond ONLY with valid JSON matching this exact schema:
{"sentiment": "POSITIVE|NEGATIVE|NEUTRAL", "confidence": 0.0-1.0, "reason": "one sentence"}

No markdown, no explanation, no extra keys. JSON only.

Review: "The product broke after two days. Terrible quality."
"""

constrained_raw = ollama_generate(constrained_prompt, temperature=0.0)
print("=== Constrained output (raw) ===")
print(constrained_raw)

print()
try:
    parsed = json.loads(constrained_raw)
    print("✅ Valid JSON parsed:", parsed)
except json.JSONDecodeError as e:
    print(f"❌ JSON parse failed: {e}")

### 2e) Side-by-side comparison of all prompt formats

In [ ]:
results = {
    "Zero-shot":   zero_shot,
    "Role prompt": role_result,
    "Few-shot":    few_shot_result,
    "Constrained": constrained_raw,
}

print(f"Task: {TASK}")
print()
print(f"{'Format':<15} {'Output'}")
print("-" * 60)
for fmt, out in results.items():
    short = out.replace("\n", " ")[:80]
    print(f"{fmt:<15} {short}")

### 2f) Drill — try your own prompt variants
Change `MY_TASK` and `MY_SYSTEM` and re-run to experiment.

In [ ]:
MY_TASK = "Summarise this in exactly one sentence: 'Vector databases store embeddings and support fast approximate nearest-neighbour search. They are used in RAG pipelines to retrieve relevant context for LLMs.'"
MY_SYSTEM = "You are a concise technical writer. One sentence only. No filler words."

my_answer = ollama_generate(MY_TASK, system=MY_SYSTEM, temperature=0.3)
print(my_answer)

---
## Part 3 — Temperature & top_p: Output Consistency

### What do these parameters do?

**Temperature** controls randomness in token sampling:
- `0.0` → always picks the highest-probability token (deterministic, repetitive)
- `0.7` → balanced creativity and coherence (good general default)
- `1.5+` → very random, often incoherent

**top_p (nucleus sampling)** limits the pool of tokens to sample from:
- `top_p=1.0` → consider all tokens
- `top_p=0.9` → only tokens whose cumulative probability ≤ 90% (removes low-prob outliers)
- `top_p=0.1` → very conservative — only the top few tokens

> Rule of thumb: tune **one at a time** — either temperature or top_p, not both simultaneously.

### 3a) Vary temperature — same prompt, 5 different settings

In [ ]:
CONSISTENCY_PROMPT = "Name one unexpected use case for machine learning."
TEMPERATURES = [0.0, 0.3, 0.7, 1.0, 1.5]

print(f"Prompt: '{CONSISTENCY_PROMPT}'")
print()
print(f"{'Temp':<8} {'Response'}")
print("-" * 70)

temp_results = {}
for temp in TEMPERATURES:
    out = ollama_generate(CONSISTENCY_PROMPT, temperature=temp, top_p=1.0)
    temp_results[temp] = out
    short = out.replace("\n", " ")[:90]
    print(f"{temp:<8} {short}")

### 3b) Consistency test — same temp, repeat 5 times
Low temp → consistent; high temp → varied each run.

In [ ]:
def consistency_test(prompt: str, temperature: float, runs: int = 5) -> None:
    print(f"Temperature={temperature} | {runs} runs")
    outputs = []
    for i in range(runs):
        out = ollama_generate(prompt, temperature=temperature, top_p=0.9)
        outputs.append(out.replace("\n", " ")[:80])
        print(f"  [{i+1}] {outputs[-1]}")

    unique = len(set(outputs))
    print(f"  → {unique}/{runs} unique responses")
    print()

REPEAT_PROMPT = "In one sentence, what is a transformer model?"
consistency_test(REPEAT_PROMPT, temperature=0.0)
consistency_test(REPEAT_PROMPT, temperature=1.0)

### 3c) Vary top_p — with temperature fixed at 0.8

In [ ]:
TOP_P_VALUES = [0.1, 0.5, 0.9, 1.0]
CREATIVE_PROMPT = "Write one line of a haiku about neural networks."

print(f"Prompt: '{CREATIVE_PROMPT}' (temperature=0.8 fixed)")
print()
print(f"{'top_p':<8} {'Response'}")
print("-" * 70)

for top_p in TOP_P_VALUES:
    out = ollama_generate(CREATIVE_PROMPT, temperature=0.8, top_p=top_p)
    short = out.replace("\n", " ")[:90]
    print(f"{top_p:<8} {short}")

### 3d) Grid sweep — temperature × top_p
Run a small grid to find the best combination for a structured task.

In [ ]:
STRUCTURED_PROMPT = (
    "Reply ONLY with a JSON object: {\"answer\": \"yes\" or \"no\", \"reason\": \"one sentence\"}. "
    "Question: Is Python interpreted?"
)

grid_temps   = [0.0, 0.5, 1.0]
grid_top_ps  = [0.5, 0.9]

print(f"{'temp':<8} {'top_p':<8} {'valid_json':<12} {'output'}")
print("-" * 80)

for temp in grid_temps:
    for tp in grid_top_ps:
        raw = ollama_generate(STRUCTURED_PROMPT, temperature=temp, top_p=tp)
        try:
            json.loads(raw)
            valid = "✅ yes"
        except json.JSONDecodeError:
            valid = "❌ no"
        short = raw.replace("\n", " ")[:55]
        print(f"{temp:<8} {tp:<8} {valid:<12} {short}")

### 3e) Latency vs temperature
Does temperature change how long generation takes?

In [ ]:
LATENCY_PROMPT = "Explain what backpropagation does in two sentences."

print(f"{'temp':<8} {'time (s)':>10}")
print("-" * 22)
for temp in [0.0, 0.5, 1.0, 1.5]:
    t0 = time.perf_counter()
    ollama_generate(LATENCY_PROMPT, temperature=temp)
    elapsed = time.perf_counter() - t0
    print(f"{temp:<8} {elapsed:>10.2f}")

---
## Part 4 — Prompt Lab: Track Everything in a Scorecard

Score each response 1–5 across four dimensions:

| Dimension | What to evaluate |
|---|---|
| **Correctness** | Factually accurate? |
| **Format** | Matches the requested output structure? |
| **Conciseness** | No unnecessary padding or repetition? |
| **Consistency** | Same result across repeated runs? |

In [ ]:
variants = [
    {
        "name": "zero-shot",
        "system": "",
        "prompt": "What is gradient descent?",
        "temperature": 0.7,
    },
    {
        "name": "role + constraint",
        "system": "You are a machine learning teacher. Answer in exactly two sentences. No bullet points.",
        "prompt": "What is gradient descent?",
        "temperature": 0.2,
    },
    {
        "name": "few-shot",
        "system": "",
        "prompt": (
            "Answer in exactly two sentences.\n\n"
            "Q: What is a neural network?\n"
            "A: A neural network is a computational model inspired by the human brain. It learns patterns from data by adjusting weighted connections between layers.\n\n"
            "Q: What is an activation function?\n"
            "A: An activation function introduces non-linearity into a neural network. It determines whether a neuron should fire based on its weighted input.\n\n"
            "Q: What is gradient descent?\n"
            "A:"
        ),
        "temperature": 0.1,
    },
]

scorecard = []
for v in variants:
    output = ollama_generate(v["prompt"], system=v["system"], temperature=v["temperature"])
    scorecard.append({"name": v["name"], "output": output})

print(f"{'Variant':<20} {'Output preview'}")
print("-" * 80)
for row in scorecard:
    preview = row["output"].replace("\n", " ")[:80]
    print(f"{row['name']:<20} {preview}")

### 4b) Score your results manually below
Replace the `None` values with your 1–5 scores after reading the outputs above.

In [ ]:
manual_scores = [
    {"name": "zero-shot",        "correctness": None, "format": None, "conciseness": None, "consistency": None},
    {"name": "role + constraint","correctness": None, "format": None, "conciseness": None, "consistency": None},
    {"name": "few-shot",          "correctness": None, "format": None, "conciseness": None, "consistency": None},
]

print(f"{'Variant':<20} {'Correctness':>13} {'Format':>8} {'Concise':>9} {'Consist':>9}")
print("-" * 65)
for s in manual_scores:
    print(
        f"{s['name']:<20}"
        f" {str(s['correctness']):>13}"
        f" {str(s['format']):>8}"
        f" {str(s['conciseness']):>9}"
        f" {str(s['consistency']):>9}"
    )

---
## Week 1 Summary

| Topic | What you did |
|---|---|
| **Tokenization** | Used `tiktoken` to inspect token splits, edge cases, emoji, special tokens |
| **Context budget** | Built a calculator to measure prompt size vs model limit |
| **Prompt formats** | Compared zero-shot, role, few-shot, constrained on the same task |
| **Temperature** | Swept 0.0 → 1.5, measured consistency + output diversity |
| **top_p** | Swept 0.1 → 1.0 with fixed temp, observed creativity shift |
| **Grid search** | Temperature × top_p grid for structured JSON output reliability |
| **Prompt scorecard** | Evaluated 3 variants manually across 4 quality dimensions |

## Week 1 Reflection
- Which prompt format gave the most consistent, structured output?
- At what temperature did JSON validity start breaking?
- What surprised you most about how tokenization splits words?